# Phase 5 — 모든 HWP·HWPX·PDF의 실제 인식·검색·요약

이 노트북은 일부 표본이나 ID 목록만 검사하지 않습니다. 기본 대상은 현재 **원본 100개 / 대표 99개 전체**입니다.
Git에서 받은 `rfp_phase5_corpus/`의 읽을 수 있는 Python 코드를 사용합니다. 먼저 최신 변경을 pull하세요.

1. 실제 파일의 SHA·magic 형식·파서 버전과 저장 추출 내용을 문서마다 연결합니다.
2. 모든 추출 원문의 청크 보존, 인덱스 내용, 실제 본문 구간 검색을 확인합니다. 기본 검색은 RRF이고 BM25 비교도 남깁니다.
3. 요약은 전체 문서를 선택한 채 이번 실행 예산만큼 처리하고 이어서 실행합니다. 전체 잔량이 크다는 이유로 API 전체를 끄지 않습니다.

**첫 Run All은 API 0회**입니다. 실제 요약은 생성 셀의 `RUN_LIVE_SUMMARIES=True` 하나로 켭니다.
기존 47개 완료 배치·누적 사용량·검토표를 보존합니다. 기존 실패는 자동으로 재시도하지 않습니다.

`REPARSE_RAW=True`는 전체 원본을 캐시 없이 다시 파싱하므로 수 분 이상 걸릴 수 있습니다.
False에서는 원본 SHA·형식·파서 버전과 저장 추출 내용을 대조하며, 새로 파싱했다고 표시하지 않습니다.
개발 검증에서는 실제 99개 고유 원본을 전수 재파싱해 기존 본문·표와 일치함을 확인했습니다.
PDF의 그림·벡터 숫자·표 해석 및 요약 정답성은 별도입니다. OCR·시각 검토가 필요한 문서를 정상 완료로 숨기지 않습니다.


In [ ]:
from pathlib import Path
import json,sys

PROJECT_ROOT = None
PARSER_ROOT = None  # Phase2 local_ingest/ 폴더가 들어 있는 프로젝트
RAW_DIRECTORY = None  # 기본: Phase2/data/raw
INPUT_DIRECTORY = None  # 새 문서 폴더를 지정하면 별도 workspace에 파싱·색인
WORKSPACE = None
REPARSE_RAW = False
ALLOW_UNREVIEWED_SOURCES = False  # OCR·부분 추출 경고를 직접 확인한 뒤에만 True
RUN_ID = 'phase5_final_source_units'
MAX_NEW_CALLS = 20
MAX_TOTAL_API_CALLS = 1000  # 실제 생성 시 적용할 명시적 누적 상한
MAX_TOTAL_INPUT_BYTES = 40_000_000
DOTENV_PATH = None

if PROJECT_ROOT is None:
    candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'rfp_phase5_local_20261111']
    BASE_ROOT=next((p.resolve() for p in candidates if (p/'rfp_phase5_final/engine.py').is_file()),None)
else:BASE_ROOT=Path(PROJECT_ROOT).expanduser().resolve()
if BASE_ROOT is None:raise FileNotFoundError('PROJECT_ROOT에 Phase5 프로젝트 폴더를 지정하세요.')
if not (BASE_ROOT/'rfp_phase5_corpus/execution.py').is_file():
    raise FileNotFoundError('git pull로 rfp_phase5_corpus 모듈을 먼저 받으세요.')
if str(BASE_ROOT) not in sys.path:sys.path.insert(0,str(BASE_ROOT))
for name,module in list(sys.modules.items()):
    if name.split('.')[0] in {'local_rag','rfp_phase4','rfp_phase5','rfp_phase5_final','rfp_phase5_corpus'}:
        if getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(BASE_ROOT):
            raise RuntimeError('다른 프로젝트 모듈이 남아 있습니다. 커널 Restart 후 실행하세요.')
intake_path=BASE_ROOT/'reports/intake_report.json'
previous_intake=json.loads(intake_path.read_text(encoding='utf-8')) if intake_path.is_file() else {}
PARSER_ROOT=Path(PARSER_ROOT).expanduser().resolve() if PARSER_ROOT else Path(previous_intake.get('parser_root') or BASE_ROOT.parent/'rfp_phase2_local_20261111')
if RAW_DIRECTORY is None and previous_intake:RAW_DIRECTORY=BASE_ROOT/'data/raw'
ROOT=BASE_ROOT
print('프로젝트:',ROOT)
print('파서:',PARSER_ROOT)
print('대상: 전체 문서 / 이번 생성 상한:',MAX_NEW_CALLS,'회')


## 1. 새 문서 입력과 기존 자료 로드

`INPUT_DIRECTORY`가 없으면 현재 100개 문서를 검사합니다. 폴더를 지정하면 HWP/PDF/HWPX를 실제로 파싱하고 새 인덱스를 만듭니다.
기존 인덱스·원문·완료 기록은 덮어쓰지 않습니다. 손상·암호화·이미지 전용 문서는 파일별로 보고하며 다른 파일 처리는 계속합니다.
다른 문서의 정답표를 새 문서에 적용하지 않습니다. SHA와 ID가 맞는 기존 시각 검수 보충만 명시적으로 보존합니다.


In [ ]:
from rfp_phase5_corpus.intake import build_workspace
from rfp_phase5.runtime import bootstrap_phase5
from rfp_phase5_corpus.execution import open_corpus_run
from rfp_phase5_corpus.workflow import audit_corpus
from rfp_phase5_corpus.index_audit import search_corpus
from rfp_phase5_final.engine import render_result,write_json
from IPython.display import display,Markdown

if INPUT_DIRECTORY is not None:
    intake=build_workspace(BASE_ROOT,INPUT_DIRECTORY,parser_root=PARSER_ROOT,workspace=WORKSPACE)
    ROOT=Path(intake['workspace']);RAW_DIRECTORY=ROOT/'data/raw'
    display(intake)
if 'app' in globals() and isinstance(app,dict) and app.get('index') is not None:app['index'].close()
app=bootstrap_phase5(ROOT)
runner=open_corpus_run(app,RUN_ID,max_total_calls=MAX_TOTAL_API_CALLS,max_total_input_bytes=MAX_TOTAL_INPUT_BYTES)
RAW_DIRECTORY=Path(RAW_DIRECTORY).expanduser().resolve() if RAW_DIRECTORY else PARSER_ROOT/'data/raw'
print('원본 파일 폴더:',RAW_DIRECTORY)
print('실행 기록:',runner.folder)


## 2. 모든 원본 → 파싱 → 청크 → 검색 전수 검사

문서별 상태는 `read_ready`, `needs_review`, `blocked`로 구분합니다.
`needs_review`는 파싱 실패가 아니라 표·숫자·OCR·구조 등에 확인할 사항이 남았다는 뜻입니다.
색인 문자 보존율은 추출 이후의 보존이며, 이미지에서 놓친 글자까지 100% 읽었다는 뜻이 아닙니다.


In [ ]:
def progress(n,total,row):
    if n%10==0 or n==total or not row['read_success']:
        print(f"원본 [{n}/{total}] {row['file_name']} / 읽기={row['read_success']}",flush=True)

corpus_audit=audit_corpus(app,parser_root=PARSER_ROOT,raw_directory=RAW_DIRECTORY,
    reparse=REPARSE_RAW,progress=progress,allow_unreviewed_sources=ALLOW_UNREVIEWED_SOURCES)
display({k:corpus_audit[k] for k in ('original_documents','canonical_documents',
    'readable_original_documents','blocked_original_documents','needs_review_original_documents',
    'unregistered_raw_sources','fresh_parsing_verified')})

def show_documents(keyword=''):
    rows=[r for r in corpus_audit['documents'] if keyword.casefold() in
        (r['file_name']+' '+r['doc_id']+' '+r['canonical_doc_id']).casefold()]
    display(rows)
    print('문서 수:',len(rows))
    return rows

all_documents=show_documents()
print('전체 원본별 결과 CSV:',ROOT/'phase5_outputs/corpus_audit/documents.csv')


## 3. 전체 문서에서 실제 검색

문서 ID를 지정하지 않으면 전체 대표 문서에서 검색합니다. ID를 지정하면 해당 범위를 보존합니다.
RRF를 기본으로 사용하며 검색 점수를 답변 신뢰도나 확률로 표시하지 않습니다.


In [ ]:
QUERY = '사업예산 부가세 포함 수행기간'
SEARCH_DOC_IDS = None  # None=전체, 또는 원본/대표 ID 목록
hits=search_corpus(app,QUERY,doc_ids=SEARCH_DOC_IDS,top_k=8)
display([{'doc_id':h['doc_id'],'file_name':h.get('file_name'),'locator':h.get('locator'),
    'score':h.get('score'),'text_preview':h['text'][:350]} for h in hits])


## 4. 전체 문서 요약 — 이번 예산만큼 처리하고 이어서 실행

모든 문서를 선택하지만 한 번에 최대 `MAX_NEW_CALLS`회만 새로 호출합니다.
전체 계획이 총예산을 넘더라도 남은 예산 내에서 부분 진행합니다. 같은 셀 재실행 시 완료 배치는 건너뜁니다.
키는 `.env`에서 읽습니다. 검사만 하려면 아래 옵션을 False로 유지하세요.
기존 실패 배치의 재시도는 별도 복구 명령으로 명시적으로 수행합니다.


In [ ]:
RUN_LIVE_SUMMARIES = False  # True이면 전체 문서 중 이번 상한만큼 실제 요약
if type(RUN_LIVE_SUMMARIES)is not bool:raise TypeError('실행 옵션은 True/False여야 합니다.')
print('이번 실제 생성 옵션:',RUN_LIVE_SUMMARIES,'/ 전체 문서 선택 / 추가 호출 상한:',MAX_NEW_CALLS)
if RUN_LIVE_SUMMARIES:
    from rfp_phase5_recovery.runner import load_api_key
    if not corpus_audit['eligible_canonical_doc_ids']:
        raise ValueError('원문·색인·검색 검사에서 실행 가능한 문서가 없습니다.')
    if MAX_NEW_CALLS>0:load_api_key(BASE_ROOT,DOTENV_PATH)
    corpus_execution=runner.run_corpus(corpus_audit['eligible_canonical_doc_ids'],
        allow_api=True,max_new_calls=MAX_NEW_CALLS)
    invocation=corpus_execution.get('invocation')
    corpus_execution=runner.summarize_corpus('all')
    corpus_execution['invocation']=invocation
    corpus_execution['generation_eligible_doc_ids']=corpus_audit['eligible_canonical_doc_ids']
    write_json(runner.folder/'corpus_execution.json',corpus_execution)
else:
    corpus_execution=runner.summarize_corpus('all')
display({k:corpus_execution[k] for k in ('selected_original_documents','selected_canonical_documents',
    'planned_batches','completed_batches','failed_batches','awaiting_batches','completed_documents',
    'all_selected_generation_completed','budget')})
print('전 문서 생성 상태:',runner.folder/'corpus_execution.json')


## 5. 문서별 생성 결과와 사람 검토

아래 함수에 임의 문서 ID를 넣으면 해당 문서 결과와 검토표 위치를 확인할 수 있습니다.
미처리 문서·검증 실패·추출 경고가 남아 있으면 전체 Phase 5 완료로 판정하지 않습니다.
실제 의미적 정확성·요약 누락·조건 보존은 생성 결과와 원문을 대조해야 합니다.


In [ ]:
def show_summary(doc_id):
    result=runner.result(doc_id)
    display(Markdown(render_result(result)))
    print('상태:',result['execution_status'],'/ 주장 검사율:',result['checked_chunk_coverage'])
    print('문서 결과·검토표 폴더:',runner.folder/'documents'/result['doc_id'])
    return result

print('전체 문서 인식 검사:',corpus_audit['all_existing_documents_text_ready'])
print('전체 원본 등록 완료:',corpus_audit['all_raw_files_registered'])
whole_generation_complete=(corpus_execution['all_canonical_documents_selected']
    and corpus_execution['all_selected_generation_completed']
    and corpus_audit['all_existing_documents_text_ready'] and corpus_audit['all_raw_files_registered'])
print('전체 요약 생성 완료:',whole_generation_complete)
print('전체 의미적 정답·시각 판독 완전성: 자동 통과 판정하지 않음')
